# Requirement 8 · Benchmark: R-CNN, Fast R-CNN, Faster R-CNN, RetinaNet, YOLO, ResNet, GoogLeNet

All detector families on the same Kaggle **test classroom**, same metric code, thresholds from validation.
R-CNN and Fast R-CNN are implemented in `src/smartattend/vision/rcnn.py`; see `docs/guide/08_benchmark.md`.

**Use a GPU runtime.** On a laptop CPU the whole benchmark takes about 2 hours (R-CNN alone needs about 11 s per frame).

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

In [ ]:
import os, kagglehub
if not os.path.exists('data/processed/kaggle_classroom/crops'):
    from smartattend.data.kaggle_classroom import prepare
    prepare('data/raw', 'data/processed/kaggle_classroom',
            source=kagglehub.dataset_download('mustafaxgm/classroom-student-engagement-dataset'))

## 1 · Region proposals (selective search)

Selective search is in OpenCV's *contrib* build. On Colab it is safe to install it directly
(on the laptop we load it from a separate folder so the project's OpenCV is untouched).

In [ ]:
!pip install -q opencv-contrib-python
!python scripts/selective_search.py

## 2 · Requirement 7 first (it fine-tunes YOLOv8n and the ResNet18 / GoogLeNet classifiers that are compared here)

In [ ]:
from smartattend.vision.finetune_yolo import finetune
import os
if not os.path.exists('models/yolo/yolov8n-kaggle.pt'):
    finetune('data/processed/kaggle_classroom/yolo_person', 'models', 'reports/experiments/detection',
             epochs=15, run_name='yolov8n_kaggle', target_name='yolov8n-kaggle.pt')
!python -m smartattend classroom

## 3 · The benchmark

In [ ]:
!python -m smartattend benchmark --tv-epochs 2

In [ ]:
from IPython.display import Image, display
display(Image('reports/figures/vision/benchmark_detectors.png'))

## 4 · Tests

In [ ]:
!python -m pytest -q tests/test_rcnn.py